# 🧪 Lab 2 — MLP com Keras/TensorFlow

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Objetivo:** Construir, treinar e salvar uma MLP com Keras para estimar o **rendimento de um reator CSTR**.

**Baseline a bater:** XGBoost RMSE ~ 1.5%.

---

**Roteiro de 10 células:** 1-4 setup+dados+baseline → 5-7 MLP Keras → 8-10 avaliação+conclusão.
Cada célula vale um item do checklist. **Anote em markdown** as decisões de arquitetura.

### Célula 1 — Importar bibliotecas

Verifique a versão do TensorFlow.

In [ ]:
# !pip install tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

print(f"TensorFlow version: {tf.__version__}")

### Célula 2 — Carregar dados

Carregue `reator_rendimento.csv`. Explore com `info()`, `describe()`, heatmap.

**Anote:** relações entre as variáveis e o rendimento?

In [ ]:
URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula10/reator_rendimento.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)

# TODO: explorar
# print(df.info())
# print(df.describe().round(2))
# sns.heatmap(df.corr(), annot=True)

### Célula 3 — Features + Split + Escalonamento

Crie lags (k=1,2) e média móvel (janela=3) para T_reator. Split 80/20. `StandardScaler`.

In [ ]:
# TODO: feature engineering + split + escala
# df['T_lag1'] = df['T_reator_C'].shift(1)
# df['T_lag2'] = df['T_reator_C'].shift(2)
# df['T_ma3'] = df['T_reator_C'].rolling(3).mean()
# df = df.dropna()
#
# feat = ['T_reator_C', 'vazao_L_min', 'pressao_bar', 'conc_alimentacao_mol_L', 'T_lag1', 'T_lag2', 'T_ma3']
# X = df[feat]
# y = df['rendimento_pct']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# scaler = StandardScaler()
# X_train_s = scaler.fit_transform(X_train)
# X_test_s = scaler.transform(X_test)

### Célula 4 — Baseline: XGBoost

Treine XGBoost. RMSE de referência ~1.5%. **A MLP precisa bater isso.**

In [ ]:
# TODO: baseline XGBoost
# xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
# xgb.fit(X_train, y_train)
# rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb.predict(X_test)))
# print(f"XGBoost baseline RMSE: {rmse_xgb:.3f}%")

### Célula 5 — Definir arquitetura (Sequential)

`Input → Dense(64,'relu') → Dropout(0.2) → Dense(32,'relu') → Dense(1)`

**Explique cada escolha em texto.**

In [ ]:
# TODO: definir arquitetura
# model = keras.Sequential([
#     layers.Input(shape=(X_train_s.shape[1],)),
#     layers.Dense(64, activation='relu'),
#     layers.Dropout(0.2),
#     layers.Dense(32, activation='relu'),
#     layers.Dense(1)
# ])
# model.summary()

### Célula 6 — Compilar + Callbacks

`Adam(lr=0.001)`, `loss='mse'`, `metrics=['mae']`.  
`EarlyStopping(patience=15)` + `ReduceLROnPlateau(factor=0.5, patience=5)`.

In [ ]:
# TODO: compilar + callbacks
# model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
#               loss='mse', metrics=['mae'])
# callbacks = [
#     keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
#     keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
# ]

### Célula 7 — Treinar + Loss Curve

`epochs=200`, `validation_split=0.1`, `batch_size=32`. Plote a curva de perda.

In [ ]:
# TODO: treinar
# history = model.fit(X_train_s, y_train, epochs=200, validation_split=0.1,
#                     callbacks=callbacks, batch_size=32, verbose=1)
#
# # Plot loss curve
# plt.figure(figsize=(8, 4))
# plt.plot(history.history['loss'], label='Treino')
# plt.plot(history.history['val_loss'], label='Validação')
# plt.xlabel('Época'); plt.ylabel('Loss (MSE)')
# plt.legend(); plt.grid(alpha=0.3); plt.show()

### Célula 8 — Avaliar no teste

`model.evaluate(X_test_s, y_test)` + `model.predict()`. Calcule RMSE.

In [ ]:
# TODO: avaliar
# test_loss, test_mae = model.evaluate(X_test_s, y_test)
# y_pred_mlp = model.predict(X_test_s).ravel()
# rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
# r2_mlp = r2_score(y_test, y_pred_mlp)
# print(f"MLP: RMSE={rmse_mlp:.3f}%  R²={r2_mlp:.3f}")

### Célula 9 — Comparar com baseline

Tabela MLP vs XGBoost. Plot predicted vs real (ambos lado a lado).

In [ ]:
# TODO: comparar
# y_pred_xgb = xgb.predict(X_test)
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# axes[0].scatter(y_test, y_pred_mlp, alpha=0.3, s=8)
# axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
# axes[0].set_title(f'MLP — RMSE: {rmse_mlp:.3f}')
# axes[1].scatter(y_test, y_pred_xgb, alpha=0.3, s=8, color='green')
# axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
# axes[1].set_title(f'XGBoost — RMSE: {rmse_xgb:.3f}')
# plt.tight_layout()
# plt.show()

### Célula 10 — Salvar modelo + Conclusão

Salve o modelo como `soft_sensor_reator.keras`. Conclusão em 3 parágrafos.

In [ ]:
# TODO: salvar modelo
# model.save('soft_sensor_reator.keras')
# print("Modelo salvo!")

> **Conclusão (3 parágrafos):**
> 1. A MLP atingiu/bateu o baseline XGBoost?
> 2. Quantas épocas até o early stopping?
> 3. O modelo está pronto para implantar como soft-sensor?

---

## Checklist de Boas Práticas Keras

- [ ] TensorFlow instalado e versão verificada
- [ ] Dataset explorado (info/describe/heatmap)
- [ ] Features + lags + média móvel criados
- [ ] Baseline XGBoost treinado
- [ ] Arquitetura definida (Input → Dense → Dropout → Dense → Dense)
- [ ] Compilado (Adam + mse + mae)
- [ ] EarlyStopping + ReduceLROnPlateau usados
- [ ] Loss curve plotada
- [ ] Comparado com XGBoost (tabela + gráfico)
- [ ] Modelo salvo (.keras)